In [2]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
import time
import json

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler

from pathlib import Path
parent_dir = Path(os.getcwd()).parent
sys.path.append(str(parent_dir))

from libML.data_load_and_label_for_training import load_emg_data
from libML.models import choose_model
from libML.evaluation import compute_scores, plot_cv_scores, plot_labels
from libML.export import save_best_params, save_model
from libML.preprocessing_new import segment_aux_windows_new, notch_filter, passband_filter
from libML.feature_engineering import extract_window_features


In [3]:
def load_model_and_scaler(model_name, dof_name, model_dir):
    """
    Load a saved model and its scaler from disk.
    
    Args:
        model_name (str): Name of the model (e.g., "LDA", "SVM")
        dof_name (str): Name of the degree of freedom (e.g., "wrist_flex_ext")
        model_dir (str): Directory where models are saved
    
    Returns:
        tuple: (model, scaler) - loaded model and scaler objects
    """
    try:
        model_path = os.path.join(model_dir, f"{model_name}/model_{dof_name}.joblib")
        scaler_path = os.path.join(model_dir, f"{model_name}/scaler_{dof_name}.joblib")
        
        # Load model
        model = joblib.load(model_path)
        print(f"Loaded model from: {model_path}")
        
        # Load scaler if it exists
        scaler = None
        if os.path.exists(scaler_path):
            scaler = joblib.load(scaler_path)
            print(f"Loaded scaler from: {scaler_path}")
        else:
            print(f"No scaler found at: {scaler_path}")
            
        return model, scaler
        
    except Exception as e:
        print(f"Error loading model or scaler for {dof_name}: {e}")
        return None, None

def measure_inference_time(model, scaler, sample_data, num_iterations=1000):
    """
    Measure inference time for a single prediction.
    
    Args:
        model: Loaded scikit-learn model
        scaler: Loaded scaler (or None)
        sample_data: Single sample for prediction (1D array)
        num_iterations (int): Number of iterations for timing
    
    Returns:
        dict: Timing results in milliseconds
    """
    # Ensure sample_data is 2D (samples, features)
    if len(sample_data.shape) == 1:
        sample_data = sample_data.reshape(1, -1)
    
    # Apply scaler if available
    if scaler is not None:
        sample_data = scaler.transform(sample_data)
    
    # Warm-up run
    _ = model.predict(sample_data)
    
    # Measure inference time
    start_time = time.perf_counter()
    for _ in range(num_iterations):
        _ = model.predict(sample_data)
    end_time = time.perf_counter()
    
    total_time_ms = (end_time - start_time) * 1000
    avg_time_ms = total_time_ms / num_iterations
    
    return {
        'total_time_ms': total_time_ms,
        'avg_time_ms': avg_time_ms,
        'num_iterations': num_iterations
    }

def run_single_inference(model, scaler, sample_data):
    """
    Run a single inference and return the prediction.
    
    Args:
        model: Loaded scikit-learn model
        scaler: Loaded scaler (or None)
        sample_data: Single sample for prediction
    
    Returns:
        prediction: Model prediction
    """
    # Ensure sample_data is 2D (samples, features)
    if len(sample_data.shape) == 1:
        sample_data = sample_data.reshape(1, -1)
    
    # Apply scaler if available
    if scaler is not None:
        sample_data = scaler.transform(sample_data)
    
    # Run prediction
    prediction = model.predict(sample_data)
    return prediction[0]  # Return single prediction

In [ ]:
parent_dir = Path(os.getcwd()).parent
MODEL_DIR = os.path.join(parent_dir, "./results/models")  # Your model directory
MODEL_NAME = "RandomForest"  # e.g., "LDA", "SVM", "RandomForest"
DOF_NAME = "wrist_flex_ext"  # Your degree of freedom namZ

# Load model and scaler
model, scaler = load_model_and_scaler(MODEL_NAME, DOF_NAME, MODEL_DIR)

if model is None:
    print("Failed to load model")
    sys.exit(1)

# Create sample data (replace with your actual feature data)
# This should match the shape your model expects
sample_data = np.random.randn(1, 126)  # Example: 1 sample with 8 features

# Run single inference
print("\n--- Single Inference ---")
prediction = run_single_inference(model, scaler, sample_data)
print(f"Prediction: {prediction}")

# Measure inference time
print("\n--- Inference Timing ---")
timing_results = measure_inference_time(model, scaler, sample_data, num_iterations=1000)
    
print(f"Total time for {timing_results['num_iterations']} iterations: {timing_results['total_time_ms']:.2f} ms")
print(f"Average inference time: {timing_results['avg_time_ms']:.4f} ms")
print(f"Potential throughput: {1000/timing_results['avg_time_ms']:.0f} inferences/second")

Loaded model from: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/model_wrist_flex_ext.joblib
No scaler found at: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/scaler_wrist_flex_ext.joblib

--- Single Inference ---
Prediction: 1

--- Inference Timing ---
Total time for 1000 iterations: 17262.36 ms
Average inference time: 17.2624 ms
Potential throughput: 58 inferences/second


In [14]:
# For testing multiple models
DOF_NAMES = ["wrist_flex_ext", "wrist_pro_sup", "thumb_flex_ext", "thumb_abd_add", 
             "index_flex_ext", "middle_flex_ext", "ring_flex_ext", "pinky_flex_ext"]  # Your DOF names
MODEL_NAMES = ["RandomForest"]  # Your model names "LDA", "SVM", "RandomForest"

# Create sample data (adjust shape to match your features)
sample_data = np.random.randn(1, 126)

for model_name in MODEL_NAMES:
    for dof_name in DOF_NAMES:
        print(f"\n{'='*50}")
        print(f"Testing: {model_name} - {dof_name}")
        print(f"{'='*50}")
        
        model, scaler = load_model_and_scaler(model_name, dof_name, MODEL_DIR)
        
        if model is not None:
            # Single inference
            prediction = run_single_inference(model, scaler, sample_data)
            print(f"Single prediction: {prediction}")
            
            # Timing
            timing_results = measure_inference_time(model, scaler, sample_data, 1000)
            print(f"Avg inference time: {timing_results['avg_time_ms']:.4f} ms")


Testing: RandomForest - wrist_flex_ext
Loaded model from: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/model_wrist_flex_ext.joblib
No scaler found at: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/scaler_wrist_flex_ext.joblib
Single prediction: 1
Avg inference time: 16.3380 ms

Testing: RandomForest - wrist_pro_sup
Error loading model or scaler for wrist_pro_sup: [Errno 2] No such file or directory: 'c:\\Users\\miski\\Desktop\\Neuro-X\\N-Pulse\\BMI-SOFT-Signal_Processing_ML\\./results/models\\RandomForest/model_wrist_pro_sup.joblib'

Testing: RandomForest - thumb_flex_ext
Loaded model from: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/model_thumb_flex_ext.joblib
No scaler found at: c:\Users\miski\Desktop\Neuro-X\N-Pulse\BMI-SOFT-Signal_Processing_ML\./results/models\RandomForest/scaler_thumb_flex_ext.joblib
Single prediction: 